1. Завантаження та нормалізація 5000 записів (видалення тегів, пунктуації).
2. Векторизація тексту за допомогою методу TF-IDF (обмеження: 5000 слів).
3. Навчання та порівняння двох моделей: LogisticRegression та MultinomialNB.
4. Вивід метрик F1-score та Accuracy.

In [1]:
import pandas as pd
import re
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, f1_score, accuracy_score

FILE_PATH = Path("IMDB Dataset.csv")
SAMPLE_SIZE = 5000
RANDOM_STATE = 42

def clean_text(text):
    """Очищення тексту від HTML-тегів, пунктуації та зайвих пробілів."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<.*?>', "", text)               # HTML теги
    text = re.sub(r'[^\w\s]', "", text)             # розділові знаки
    text = text.lower().strip()                     # регістр та пробіли
    return text

def load_and_prepare_data(path, size=5000):
    """Завантаження та первинна підготовка датасету."""
    if not path.exists():
        raise FileNotFoundError(f"Файл не знайдено: {path}")
    
    print(f"Завантаження даних...")
    df = pd.read_csv(path).head(size)
    
    print("Очищення тексту...")
    df["review"] = df["review"].apply(clean_text)
    
    # міняємо positive/negative на 1/0.
    df["sentiment"] = df["sentiment"].replace({"positive": 1, "negative": 0})

    # видаляємо пусті рядки
    df["sentiment"] = pd.to_numeric(df["sentiment"], errors='coerce')

    # кажемо, що це цілі числа
    df = df.dropna(subset=["sentiment"])
    
    return train_test_split(
        df["review"], df["sentiment"], 
        test_size=0.2, 
        random_state=RANDOM_STATE
    )

def evaluate_model(model, name, X_train, X_test, y_train, y_test):
    """Навчання та вивід результатів моделі."""
    print(f"\nНавчання моделі: {name}...")

    # прогноз даних, які модель ще не бачила
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    f1 = f1_score(y_test, predictions)
    acc = accuracy_score(y_test, predictions)
    
    print(f"{name} результати:")
    print(f"   - F1-score: {f1:.2%}")
    print(f"   - Accuracy: {acc:.2%}")
    print(f"\nДетальний звіт для {name}:\n")
    print(classification_report(y_test, predictions, target_names=["Negative", "Positive"]))
    
    return f1

def main():
    try:
        # підготовка даних
        X_train_raw, X_test_raw, y_train, y_test = load_and_prepare_data(FILE_PATH, SAMPLE_SIZE)

        # векторизація
        print("\nВекторизація тексту...")
        vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
        X_train = vectorizer.fit_transform(X_train_raw)
        X_test = vectorizer.transform(X_test_raw)

        # порівняння моделей
        models = [
            (LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), "Logistic Regression"),
            (MultinomialNB(), "Naive Bayes")
        ]

        for model, name in models:
            evaluate_model(model, name, X_train, X_test, y_train, y_test)
            print("-" * 50)

    except Exception as e:
        print(f"Помилка: {e}")

if __name__ == "__main__":
    main()

Завантаження даних...
Очищення тексту...

Векторизація тексту...

Навчання моделі: Logistic Regression...
Logistic Regression результати:
   - F1-score: 85.60%
   - Accuracy: 86.10%

Детальний звіт для Logistic Regression:

              precision    recall  f1-score   support

    Negative       0.89      0.85      0.87       530
    Positive       0.83      0.88      0.86       470

    accuracy                           0.86      1000
   macro avg       0.86      0.86      0.86      1000
weighted avg       0.86      0.86      0.86      1000

--------------------------------------------------

Навчання моделі: Naive Bayes...
Naive Bayes результати:
   - F1-score: 81.90%
   - Accuracy: 83.20%

Детальний звіт для Naive Bayes:

              precision    recall  f1-score   support

    Negative       0.83      0.85      0.84       530
    Positive       0.83      0.81      0.82       470

    accuracy                           0.83      1000
   macro avg       0.83      0.83      0.83  

Короткі результати тестування
Logistic Regression: продемонструвала вищу ефективність (F1-score - 85.6%).
Naive Bayes: показав дещо нижчу точність (F1-score - 81.9%).